In [26]:
# =========================================================
# FEATURE ENGINEERING + MODEL BUILDING
# YouTube Comments Sentiment Analysis
# =========================================================

# -------------------------------
# 1. Import Libraries
# -------------------------------

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB

from scipy.sparse import hstack

from textblob import TextBlob # New import for sentiment generation

# -------------------------------
# 2. Load Dataset
# -------------------------------

df = pd.read_csv("youtube_comments_preprocessed.csv")

print("Dataset Shape:", df.shape)

# -------------------------------
# 3. Display Columns
# -------------------------------

print("\nColumns in Dataset:")
print(df.columns)

# -------------------------------
# 4. Remove Missing Values
# -------------------------------

df = df.dropna()

print("\nAfter Removing Missing Values:", df.shape)

# -------------------------------
# 5. Select Features and Target
# -------------------------------
# Change column names if needed

X_text = df["text_clean"]

# Numerical Features
numerical_features = df[
    ["word_count", "char_count"]
]

# -------------------------------
# 5.1 Generate Sentiment (New Section)
# -------------------------------
# Using TextBlob for sentiment analysis as 'sentiment_encoded' was missing.
# If you have an existing sentiment column or preferred method, please replace this.
def get_sentiment_blob(text):
    analysis = TextBlob(text)
    # Classify as positive, negative, or neutral
    if analysis.sentiment.polarity > 0:
        return 1
    elif analysis.sentiment.polarity < 0:
        return -1
    else:
        return 0

df['sentiment_encoded'] = df['text_clean'].apply(get_sentiment_blob)

# Target Column
y = df["sentiment_encoded"]

# -------------------------------
# 6. TF-IDF Feature Engineering
# -------------------------------

tfidf = TfidfVectorizer(
    max_features=3000,
    ngram_range=(1,2),
    min_df=2,
    max_df=0.9,
    stop_words='english'
)

X_tfidf = tfidf.fit_transform(X_text)

print("\nTF-IDF Shape:", X_tfidf.shape)

# -------------------------------
# 7. Scale Numerical Features
# -------------------------------

scaler = StandardScaler()

X_num = scaler.fit_transform(numerical_features)

# -------------------------------
# 8. Combine Features
# -------------------------------

X = hstack([X_tfidf, X_num])

print("Final Feature Matrix Shape:", X.shape)

# -------------------------------
# 9. Train-Test Split
# -------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTraining Shape:", X_train.shape)
print("Testing Shape:", X_test.shape)

# =========================================================
# MODEL BUILDING
# =========================================================

# -------------------------------
# 10. Logistic Regression
# -------------------------------

lr_model = LogisticRegression(
    max_iter=1000
)

lr_model.fit(X_train, y_train)

lr_pred = lr_model.predict(X_test)

print("\n===== Logistic Regression =====")

print("Accuracy:", accuracy_score(y_test, lr_pred))

print(classification_report(
    y_test,
    lr_pred,
    zero_division=0
))

# -------------------------------
# 11. Support Vector Machine
# -------------------------------

svm_model = LinearSVC(
    C=1.5,
    class_weight='balanced',
    random_state=42
)

svm_model.fit(X_train, y_train)

svm_pred = svm_model.predict(X_test)

print("\n===== SVM Model ====")

print("Accuracy:", accuracy_score(y_test, svm_pred))

print(classification_report(
    y_test,
    svm_pred,
    zero_division=0
))

# -------------------------------
# 12. Naive Bayes
# -------------------------------

# NB works best with TF-IDF only

X_train_nb, X_test_nb, y_train_nb, y_test_nb = train_test_split(
    X_tfidf,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

nb_model = MultinomialNB()

nb_model.fit(X_train_nb, y_train_nb)

nb_pred = nb_model.predict(X_test_nb)

print("\n===== Naive Bayes =====")

print("Accuracy:", accuracy_score(y_test_nb, nb_pred))

print(classification_report(
    y_test_nb,
    nb_pred,
    zero_division=0
))

# =========================================================
# 13. Cross Validation (SVM)
# =========================================================

scores = cross_val_score(
    svm_model,
    X,
    y,
    cv=5
)

print("\n===== Cross Validation =====")

print("Cross Validation Scores:", scores)

print("Average Accuracy:", scores.mean())

# =========================================================
# 14. Predict Custom Comment
# =========================================================

sample_text = ["This video is amazing and very informative"]

# TF-IDF transform
sample_tfidf = tfidf.transform(sample_text)

# Numerical features
sample_num = pd.DataFrame({
    "word_count": [len(sample_text[0].split())],
    "char_count": [len(sample_text[0])]
})

# Scale numerical features
sample_num_scaled = scaler.transform(sample_num)

# Combine features
sample_final = hstack([sample_tfidf, sample_num_scaled])

# Prediction using SVM
prediction = svm_model.predict(sample_final)

# Label Mapping
label_map = {
    -1: "Negative",
     0: "Neutral",
     1: "Positive"
}

print("\nPredicted Sentiment:", label_map[int(prediction[0])])

print("\nFeature Engineering and Model Building Completed Successfully")

Dataset Shape: (16487, 17)

Columns in Dataset:
Index(['channel', 'text', 'like_count', 'reply_count', 'year', 'month',
       'day_of_week', 'hour', 'text_clean', 'word_count', 'char_count',
       'is_reply', 'channel_freq', 'author_comment_count',
       'video_comment_count', 'like_count_log', 'reply_count_log'],
      dtype='object')

After Removing Missing Values: (16487, 17)

TF-IDF Shape: (16487, 3000)
Final Feature Matrix Shape: (16487, 3002)

Training Shape: (13189, 3002)
Testing Shape: (3298, 3002)

===== Logistic Regression =====
Accuracy: 0.8332322619769558
              precision    recall  f1-score   support

          -1       0.84      0.32      0.47       317
           0       0.81      0.92      0.86      1417
           1       0.86      0.86      0.86      1564

    accuracy                           0.83      3298
   macro avg       0.84      0.70      0.73      3298
weighted avg       0.84      0.83      0.82      3298


===== SVM Model ====
Accuracy: 0.84111582